# Metabolomics Analyses of the Longitudinal Acne Study
## CTF and RPCA Plots

Date created: 10/18/2024

Notebook author: Britta De Pessemier 

Data analysis by: Tyler Myers, Britta De Pessemier, and Yang Chen

This notebook plots the following:
- CTF plots 
- RPCA plots

In [2]:
# Import essential Python packages
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Ellipse
from adjustText import adjust_text

# Scientific and statistical packages
import scipy.stats as ss
from skbio import DistanceMatrix
from skbio.stats.ordination import OrdinationResults
from skbio.stats.distance import permanova

# BIOM and Qiime2 related packages
import biom
from biom import load_table
from qiime2 import Artifact

# Gemelli RPCA and rarefaction utilities
from gemelli.rpca import rpca # Optional: import auto_rpca if you use auto_rpca later
from gemelli.utils import qc_rarefaction


In [5]:
# Function to draw an ellipse based on the covariance of the points
def draw_ellipse(mean, cov, ax, color, label=None, scale_factor=1.5):
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    order = eigenvalues.argsort()[::-1]
    eigenvalues = eigenvalues[order]
    eigenvectors = eigenvectors[:, order]
    
    # Calculate the angle between the x-axis and the largest eigenvector
    angle = np.degrees(np.arctan2(*eigenvectors[:, 0][::-1]))
    
    # Width and height of the ellipse (scaling applied)
    width, height = 2 * np.sqrt(eigenvalues) * scale_factor
    
    # Create the ellipse and add it to the plot
    ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
    ax.add_patch(ell)

In [3]:
# Define the color palette for the groups
group_colors = {
    "Healthy": "#58a35b",  # Color for Healthy
    "Acne_L": "#dd7966",   # Color for Acne Lesional
    "Acne_NL": "#6ab2bd"   # Color for Acne Non-lesional
}

# Map the old group names to new ones for the legend
group_name_mapping = {
    "Healthy": "Healthy",
    "Acne_L": "Acne Lesional",
    "Acne_NL": "Acne Non-lesional"
}

In [7]:
# Load the distance matrix artifact
table = Artifact.load(f'../Data/metabolomics/ctf-results_blancsremoved_09292024/distance_matrix.qza')
    
# View as a skbio DistanceMatrix object
distance_matrix = table.view(DistanceMatrix)
    
# Convert the skbio DistanceMatrix to a Pandas DataFrame
data = pd.DataFrame(distance_matrix.data, index=distance_matrix.ids, columns=distance_matrix.ids)

plot_title = 'CTF - Metabolomics '    
# Display the DataFrame
print(data)
    
# Load the subject_biplot artifact
biplot = Artifact.load(f'../Data/metabolomics/ctf-results_blancsremoved_09292024/subject_biplot.qza')
    
# View as an OrdinationResults object
ordination = biplot.view(OrdinationResults)
    
# Extract the sample coordinates as a DataFrame
sample_coordinates = pd.DataFrame(ordination.samples, index=ordination.samples.index)
    
# Extract the feature (or variable) coordinates as a DataFrame
feature_coordinates = pd.DataFrame(ordination.features, index=ordination.features.index)
    
# Display the DataFrames
print("Sample Coordinates:")
print(sample_coordinates)
print("\nFeature Coordinates:")
print(feature_coordinates)
    
# Extract the proportion explained
proportion_explained = ordination.proportion_explained
    
# Convert to percentages for PC1 and PC2
pc1_explained_variance = proportion_explained[0] * 100
pc2_explained_variance = proportion_explained[1] * 100
    
print(f"PC1 explains {pc1_explained_variance:.2f}% of the variance")
print(f"PC2 explains {pc2_explained_variance:.2f}% of the variance")
    
# Rename the sample coordinates DataFrame to spca_df
spca_df = sample_coordinates
    
# Load the metadata from the subject-metadata.tsv file
metadata_df = pd.read_csv(f'../Data/metabolomics/ctf-results_blancsremoved_09292024/subject-metadata.tsv', sep='\t')
    
# Map the index of spca_df (sample names) to the SampleID in metadata_df to get the 'group'
spca_df["Group"] = spca_df.index.map(metadata_df.set_index("#SampleID")["group"])
    
# Rename the columns of spca_df for clarity (PC1, PC2, PC3)
spca_df.columns = ['PC1', 'PC2', 'PC3', 'Group']
    
# Display the updated spca_df
print(spca_df)
    
# Plotting
mm = 1/25.4
fig, ax = plt.subplots(1, 1, figsize=(90*mm, 110*mm))
    
    # Create scatter plot with different colors based on the Group column
for group_name, group in spca_df.groupby('Group'):
    sns.scatterplot(
        data=group,
        x="PC1",
        y="PC2",
        facecolor=group_colors[group_name],  # Semi-transparent fill color
        edgecolor=group_colors[group_name],  # Colored border
        alpha=0.8,  # Transparency for the fill
        linewidth=0.5,  # Thickness of the borders
        ax=ax,
        label=group_name_mapping[group_name]
    )
    
    # Calculate mean and covariance matrix for the group
    points = group[["PC1", "PC2"]].values
    mean = np.mean(points, axis=0)
    cov = np.cov(points, rowvar=False)
    
    # Draw the ellipse for the group (with scale factor applied)
    draw_ellipse(mean, cov, ax, group_colors[group_name], scale_factor=2)
    
    # Add a subtle star (8-pointed) at the mean location for each group
    ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)
    
# Add axis labels and include the explained variance percentages
ax.set_xlabel(f'PC1 ({pc1_explained_variance:.2f}%)', fontsize=7)
ax.set_ylabel(f'PC2 ({pc2_explained_variance:.2f}%)', fontsize=7)
    
# Simplify the ticks to avoid overcrowding
yticklabels = [-0.4, -0.2, 0.0, 0.2, 0.4, 0.6]
ax.set_yticks(yticklabels)
ax.set_yticklabels(yticklabels, fontsize=7)
    
xticklabels = [-0.4, -0.2, 0.0, 0.2, 0.4, 0.6]
ax.set_xticks(xticklabels)
ax.set_xticklabels(xticklabels, fontsize=7)
    
# Add the title consistently in the top-left corner using axes coordinates
ax.text(0.05, 1.10, plot_title, fontsize=8, va='top', ha='left', transform=ax.transAxes)

# Customize legend
plt.legend(
    frameon=False,
    fontsize=7,
    title_fontsize=7
)
    
# Adjust plot style and save
ax.spines["right"].set_visible(False)
ax.spines["top"].set_visible(False)
    
plt.tight_layout()
plt.savefig(f"../Figures/metabolomics_Figures/CTF_metabolomics.png", dpi=600)
plt.show()


                   LAMI.RD003.D14.C1  LAMI.RD003.D0.C1  LAMI.RD003.D28.C1  \
LAMI.RD003.D14.C1           0.000000          4.807111           5.355293   
LAMI.RD003.D0.C1            4.807111          0.000000           0.727936   
LAMI.RD003.D28.C1           5.355293          0.727936           0.000000   
LAMI.RD001.D14.C2          26.676458         29.715920          29.814186   
LAMI.RD001.D0.C2           26.172078         28.684614          28.705186   
...                              ...               ...                ...   
LAMI.RD319.D9.C1           37.212104         34.666423          34.849380   
LAMI.RD319.D16.C1          34.837607         33.177789          33.506007   
LAMI.RD319.D21.C1          41.216455         37.999184          38.031998   
LAMI.RD319.D14.C1          33.825509         31.382228          31.577648   
LAMI.RD319.D28.C1          38.140265         35.127229          35.207422   

                   LAMI.RD001.D14.C2  LAMI.RD001.D0.C2  LAMI.RD001.D28.C2  

/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_61616/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_61616/3800277603.py:84: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ((8, 2, 0)).  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_61616/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3

## RPCA 

In [8]:
# Set the random seed for reproducibility
np.random.seed(123)
# Define the color palette for the groups
group_colors = {
    "Healthy": "#58a35b",  # Color for Healthy
    "Acne_L": "#dd7966",   # Color for Acne Lesional
    "Acne_NL": "#6ab2bd"   # Color for Acne Non-lesional
}

# Map the old group names to new ones for the legend
group_name_mapping = {
    "Healthy": "Healthy",
    "Acne_L": "Acne Lesional",
    "Acne_NL": "Acne Non-lesional"
}

In [12]:
# Load the biom table and perform RPCA
biom_tbl = biom.load_table('../Data/metabolomics/quantwithoutblancs_09292024.biom')
    
# Perform RPCA with auto rank estimation
np.seterr(divide='ignore')
ordination, distance = rpca(biom_tbl)

# Read the metadata file
metadata_file = '../Metadata/metadata_final_18062024.tsv'
metadata_df = pd.read_csv(metadata_file, sep='\t')

# Extract and view sample ordinations from RPCA result
spca_df = pd.DataFrame(ordination.samples)
spca_df["Group"] = spca_df.index.map(metadata_df.set_index("SampleID")["group"])
    
# calculate permanova F-statistic
pnova_res = permanova(distance, spca_df, "Group")
p_value = pnova_res['p-value']
    
# Plotting
mm = 1/25.4
fig, ax = plt.subplots(1, 1, figsize=(90*mm, 110*mm))
    
# Create scatter plot with different colors based on the Group column
for group_name, group in spca_df.groupby('Group'):
    sns.scatterplot(
        data=group,
        x="PC1",
        y="PC2",
        facecolor=group_colors[group_name],  # Semi-transparent fill color
        edgecolor=group_colors[group_name],  # Colored border
        alpha=0.8,  # Transparency for the fill
        linewidth=0.5,  # Thickness of the borders
        ax=ax,
        label=group_name_mapping[group_name]
    )
    # Calculate mean and covariance matrix for the group
    points = group[["PC1", "PC2"]].values
    mean = np.mean(points, axis=0)
    cov = np.cov(points, rowvar=False)
    
    # Draw the ellipse for the group (with scale factor applied)
    draw_ellipse(mean, cov, ax, group_colors[group_name], scale_factor=2)
    
    # Add a subtle star (8-pointed) at the mean location for each group
    ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)
    
# Add axis labels and include the explained variance percentages
proportion_explained = ordination.proportion_explained
pc1_explained_variance = proportion_explained[0] * 100
pc2_explained_variance = proportion_explained[1] * 100
    
ax.set_xlabel(f'PC1 ({pc1_explained_variance:.2f}%)', fontsize=7)
ax.set_ylabel(f'PC2 ({pc2_explained_variance:.2f}%)', fontsize=7)
    
# Simplify the ticks to avoid overcrowding
yticklabels = [-0.2, -0.1, 0.0, 0.1, 0.2, 0.3]
ax.set_yticks(yticklabels)
ax.set_yticklabels(yticklabels, fontsize=7)
    
xticklabels = [-0.2, -0.1, 0.0, 0.1, 0.2, 0.3]
ax.set_xticks(xticklabels)
ax.set_xticklabels(xticklabels, fontsize=7)
    
# Add the title based on the region being analyzed
ax.text(0.05, 1.10, f'RPCA - Metabolomics', fontsize=8, va='top', ha='left', transform=ax.transAxes)
    
# Add the p-value from PERMANOVA
ax.text(-0.18, 0.27, 'PERMANOVA', fontsize=7)
ax.text(-0.18, 0.25, f'***p-val = {p_value:.3f}', fontsize=7)

# Customize legend in the top-right corner
plt.legend(
    frameon=False,
    fontsize=7,
    title_fontsize=7,
    loc='upper right'
)
    
# Adjust plot style and save
ax.spines["right"].set_visible(False)
ax.spines["top"].set_visible(False)
    
plt.tight_layout()
plt.savefig(f"../Figures/metabolomics_Figures/RPCA_metabolomics.png", dpi=600)
plt.show()

/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_61616/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_61616/1302614482.py:46: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ((8, 2, 0)).  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_61616/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3

In [3]:
# Read the metadata file
metadata_file = '../Metadata/metadata_final_18062024.tsv'
metadata_df = pd.read_csv(metadata_file, sep='\t')

# Specify the group column
group_column = 'group' 

# Calculate the count of local_lesion_severity for each group
severity_counts = (
    metadata_df.groupby([group_column, 'local_lesion_severity'])
    .size()
    .unstack(fill_value=0)
)

# Print the result
print(severity_counts)

# Create a mapping for severity levels
severity_mapping = {
    0: 'absent',      # You can change 'none' to another term if desired
    1: 'low',
    2: 'low',
    3: 'moderate',
    4: 'moderate',
    5: 'high',
    6: 'high'
}

# Add the severity_level column to metadata_df
metadata_df['severity_level'] = metadata_df['local_lesion_severity'].map(severity_mapping)

# Print the updated metadata DataFrame
print(metadata_df)

# Create the combined column for severity and group
metadata_df['severity_group'] = metadata_df['severity_level'] + ' ' + metadata_df['group']


local_lesion_severity   0   1   2   3   4   5  6
group                                           
Acne_L                  0  38  32  36  28  19  6
Acne_NL                27  18   5   0   0   0  0
Healthy                57   0   0   0   0   0  0
              SampleID c_zone  \
0    LAMI.RD308.D16.C1     C1   
1    LAMI.RD310.D21.C1     C1   
2    LAMI.RD305.D21.C3     C3   
3    LAMI.RD306.D18.C2     C2   
4     LAMI.RD306.D7.C2     C2   
..                 ...    ...   
261  LAMI.RD317.D21.C1     C1   
262   LAMI.RD001.D0.C1     C1   
263  LAMI.RD014.D14.C2     C2   
264   LAMI.RD314.D0.C1     C1   
265  LAMI.RD001.D14.C2     C2   

    visual_assessment_in_vivo_number_of_non_inflammatory_lesions_face  \
0                                       not applicable                  
1                                                   72                  
2                                                   69                  
3                                       not applicable            

In [7]:
# Set the random seed for reproducibility
np.random.seed(123)

# Define the color palette for the groups
severity_group_colors = {
    "absent Healthy": "#006f00",
    "absent Acne_NL": "#9af09b",
    "low Acne_L": "#ff676c",
    "moderate Acne_L": "#ca0000", 
    "high Acne_L": "#960000"
}

# Define the shape for each group
group_shape_mapping = {
    "Healthy": "o",    # Circle
    "Acne_NL": "^",    # Triangle
    "Acne_L": "D",     # Star
}

# Define the order for the legend
severity_order = ["absent Healthy", "absent Acne_NL", "low Acne_NL", "low Acne_L", "moderate Acne_L", "high Acne_L"]


# First, filter the metadata to get the samples to be removed
samples_to_remove = metadata_df[metadata_df['severity_group'] == 'low Acne_NL']['SampleID'].tolist()

# Function to filter the biom table based on the samples to keep
def filter_biom_table(biom_table, samples_to_remove):
    # Get all sample IDs in the biom table
    all_samples = biom_table.ids(axis='sample')
    
    # Filter the samples, excluding the ones to be removed
    samples_to_keep = [sample for sample in all_samples if sample not in samples_to_remove]
    
    # Subset the biom table to only include the samples to keep
    filtered_biom_table = biom_table.filter(samples_to_keep, axis='sample', inplace=False)
    
    return filtered_biom_table, all_samples, samples_to_keep

 
# Load the biom table
#biom_tbl = biom.load_table(biom_file)
biom_tbl = biom.load_table('../Data/metabolomics/quantwithoutblancs_09292024.biom')
    
# Filter the biom table to remove unwanted samples
biom_tbl_filtered, original_samples, filtered_samples = filter_biom_table(biom_tbl, samples_to_remove)
    
# Check the samples that were removed
dropped_samples = set(original_samples) - set(filtered_samples)
    
print(f"Number of samples in original table: {len(original_samples)}")
print(f"Number of samples after filtering: {len(filtered_samples)}")
print(f"Samples that were removed: {dropped_samples}")
    
# Verify if the dropped samples match the expected samples
if set(samples_to_remove) == dropped_samples:
    print(f"Correct samples were dropped: {len(dropped_samples)} samples removed.")
else:
    print("There is a mismatch in the samples dropped. Please check!")
    
# Use filtered biom table for RPCA
np.seterr(divide='ignore')
ordination, distance = rpca(biom_tbl_filtered)

# Clear the figure to avoid overlapping plots
plt.clf()
    
# Extract and view sample ordinations from RPCA result
spca_df = pd.DataFrame(ordination.samples)
spca_df["severity_group"] = spca_df.index.map(metadata_df.set_index("SampleID")["severity_group"])
spca_df["group"] = spca_df.index.map(metadata_df.set_index("SampleID")["group"])  # Add group mapping
    
# Calculate permanova F-statistic
pnova_res = permanova(distance, spca_df, "severity_group")
p_value = pnova_res['p-value']
    
# Plotting
mm = 1/25.4
fig, ax = plt.subplots(1, 1, figsize=(90*mm, 110*mm))
    
# Create scatter plot with different colors and shapes
for severity_group in severity_order:
    group = spca_df[spca_df['severity_group'] == severity_group]
    for group_name, group_data in group.groupby('group'):
        sns.scatterplot(
            data=group_data,
            x="PC1",
            y="PC2",
            facecolor=severity_group_colors[severity_group],
            edgecolor=severity_group_colors[severity_group],
            alpha=0.8,
            linewidth=0.5,
            ax=ax,
            label=severity_group,
            marker=group_shape_mapping[group_name],  # Use shape mapping for each group
            s=10
        )
            
        # Calculate mean and covariance matrix for the group if more than 2 points
        points = group_data[["PC1", "PC2"]].values
        mean = np.mean(points, axis=0)
            
        if len(group_data) > 2:
            cov = np.cov(points, rowvar=False)
            draw_ellipse(mean, cov, ax, severity_group_colors[severity_group], scale_factor=2)

            # Add a subtle star (8-pointed) at the mean location for each group
        ax.scatter(mean[0], mean[1], color=severity_group_colors[severity_group], 
                    marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)

# Add axis labels and include the explained variance percentages
proportion_explained = ordination.proportion_explained
pc1_explained_variance = proportion_explained[0] * 100
pc2_explained_variance = proportion_explained[1] * 100
    
ax.set_xlabel(f'PC1 ({pc1_explained_variance:.2f}%)', fontsize=7)
ax.set_ylabel(f'PC2 ({pc2_explained_variance:.2f}%)', fontsize=7)

# Simplify the ticks to avoid overcrowding
yticklabels = [-0.2, -0.1, 0.0, 0.1, 0.2, 0.3]
ax.set_yticks(yticklabels)
ax.set_yticklabels(yticklabels, fontsize=7)
    
xticklabels = [-0.2, -0.1, 0.0, 0.1, 0.2, 0.3]
ax.set_xticks(xticklabels)
ax.set_xticklabels(xticklabels, fontsize=7)
    
# Add the title based on the region being analyzed
ax.text(0.05, 1.10, f'RPCA - Metabolomics', fontsize=8, va='top', ha='left', transform=ax.transAxes)
    
# Add the p-value from PERMANOVA
ax.text(-0.18, 0.27, 'PERMANOVA', fontsize=7)
ax.text(-0.18, 0.25, f'***p-val = {p_value:.3f}', fontsize=7)

# Customize legend in the top-right corner
plt.legend(
    frameon=False,
    fontsize=7,
    title_fontsize=7,
    loc='upper right'
)
    
# Adjust plot style and save
ax.spines["right"].set_visible(False)
ax.spines["top"].set_visible(False)
    
plt.tight_layout()
plt.savefig(f"../Figures/metabolomics_Figures/RPCA_Metabolomics_severity_group.png", dpi=600)
plt.show()


Number of samples in original table: 266
Number of samples after filtering: 243
Samples that were removed: {'LAMI.RD314.D28.C3', 'LAMI.RD317.D21.C3', 'LAMI.RD318.D9.C3', 'LAMI.RD318.D28.C3', 'LAMI.RD310.D7.C3', 'LAMI.RD314.D14.C3', 'LAMI.RD306.D0.C3', 'LAMI.RD319.D28.C3', 'LAMI.RD310.D21.C3', 'LAMI.RD317.D28.C3', 'LAMI.RD310.D28.C3', 'LAMI.RD314.D21.C3', 'LAMI.RD310.D0.C3', 'LAMI.RD306.D28.C3', 'LAMI.RD308.D9.C3', 'LAMI.RD304.D7.C3', 'LAMI.RD308.D21.C3', 'LAMI.RD304.D28.C3', 'LAMI.RD306.D7.C3', 'LAMI.RD319.D14.C3', 'LAMI.RD319.D7.C3', 'LAMI.RD310.D14.C3', 'LAMI.RD319.D21.C3'}
Correct samples were dropped: 23 samples removed.


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_72664/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_72664/1298537106.py:108: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ((8, 2, 0)).  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  ax.scatter(mean[0], mean[1], color=severity_group_colors[severity_group],
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_72664/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor re